# Ahzab reel: free GPU pipeline (Kaggle)

Makes everything the reel needs from **free, open models**:

| Stage | Model | Licence |
|---|---|---|
| A. Portrait of CHR-027 (4 candidates) | FLUX.1 Kontext [dev], 4-bit | FLUX.1 [dev] **non-commercial** licence |
| B. First frame of every shot, from the approved portrait | FLUX.1 Kontext [dev] | same |
| C. Image-to-video shots (breath, blinks, eye darts, expression) | LTX-Video 0.9.7 distilled | LTX-Video open weights licence |
| D. Arabic narration | Microsoft Edge neural voice `ar-SA-HamedNeural` via `edge-tts` | free, unofficial route |

**Settings before you run** (right-hand panel): Accelerator **GPU T4 x2** (or P100), Internet **On** (needs a phone-verified Kaggle account).
**Secret:** Add-ons → Secrets → `HF_TOKEN` = a Hugging Face read token, after you click *Agree* on https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev.

Run the cells top to bottom. The notebook **stops twice on purpose**: after the portraits (pick one) and after the first frames (check the face matches). Every stage skips files it already made, so re-running is safe.
When done, *Save Version → Save & Run All* is not needed: just download `ahzab_out.zip` from the Output panel and upload it to the repo at `reels/ghazwat-al-ahzab/reel2/incoming/`.

In [ ]:
!pip -q install -U "diffusers>=0.34" "transformers>=4.51" accelerate bitsandbytes sentencepiece protobuf edge-tts imageio imageio-ffmpeg

In [ ]:
import os, json, gc, math, zipfile, torch, numpy as np
from pathlib import Path
from PIL import Image
SHOTS = json.loads(r'''{
 "portrait_prompt": "One standalone front reference portrait, chest up, of the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Front-facing, both eyes into the lens, shoulders square, neutral eye level. Expression: exhausted resilience, mouth closed, no smile. Background: softly drawn black basalt field and a few date palms under an overcast sky. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
 "narration_voice": "ar-SA-HamedNeural",
 "narration": [
  {
   "id": "N1",
   "start": 8.0,
   "text": "قبلها بأيام… كان يحفر، في بردٍ قارس."
  },
  {
   "id": "N2",
   "start": 11.7,
   "text": "ينقلُ الترابَ على كتفه… وعشرةُ آلافٍ قادمون."
  },
  {
   "id": "N3",
   "start": 15.7,
   "text": "ثم جاءت الليلة… نيرانُهم في كلِّ اتجاه."
  },
  {
   "id": "N4",
   "start": 31.8,
   "text": "وقبل الفجر… أرسل اللهُ الريح."
  }
 ],
 "shots": [
  {
   "id": "B01",
   "start": 0.0,
   "dur": 4.2,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Extreme close-up of his eyes and brow at night, filling the frame. Many distant orange campfires are reflected as small points of light in both pupils; cold blue moonlight on the skin, faint warm rim from the fires, breath steam drifting past. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "His wide eyes dart slowly left, then right, following distant fires; one slow blink; steam from his breath drifts across the frame; imperceptible push-in toward the eyes. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B02",
   "start": 4.2,
   "dur": 3.7,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Close-up of his whole face at night, crouched at the edge of a freshly dug trench; behind him, out of focus across the trench, a plain full of campfires. Cold moonlight, orange fire rim light. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "Fear without melodrama: shallow quick breaths, he swallows hard, his jaw tightens, his eyes flick toward the fires and back; head cloth tail stirs in the wind; camera locked. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B03",
   "start": 7.9,
   "dur": 3.7,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Grey overcast winter morning. Medium shot: he swings a simple wooden-handled iron digging tool into hard rocky earth at the bottom of a shallow trench, dust and small stones rising; sweat on his forehead, breath visible in the cold air. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "He strikes the ground once, heavily, dust bursts up; he exhales a cloud of breath and wipes his brow with his forearm, exhausted; camera slowly drifts sideways. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B04",
   "start": 11.6,
   "dur": 4.0,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Late afternoon, overcast. Medium close-up: he carries a heavy woven palm basket of soil on his right shoulder beside a long trench, then has stopped and looks off toward the far horizon where tall columns of dust rise. Low cold sun from camera-left. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "His tired face turns slowly toward the horizon; the brows draw together, the eyes narrow and stay fixed; soil trickles from the basket; dread settles in; slow push-in. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B05",
   "start": 15.6,
   "dur": 4.1,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Night. Medium shot from the side: he kneels low at the edge of a wide trench, one hand on the cold ground; across the trench a whole plain is covered with hundreds of campfires. Deep navy sky, firelight dots. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "His eyes jump between clusters of fires, shoulders tense and raised, breath short and visible; the fires flicker; very slow pull-back revealing more fires. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B06",
   "start": 19.7,
   "dur": 5.9,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Night. Close-up of his face, eyes closed, head slightly lowered, moonlight on one side of the face and faint fire glow on the other. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "Eyes closed, his lips move silently in prayer; his fast breathing gradually slows; the clenched jaw slowly relaxes; a slow deep breath; camera very slowly pushes in. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B07",
   "start": 25.6,
   "dur": 6.1,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Night. Close-up of his face, eyes now open, looking straight ahead past the camera with a steady calm gaze; wind beginning to pull at the tail of his head cloth. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "He opens his eyes fully and holds a steady, calm gaze forward; one slow blink; the wind strengthens and whips the head cloth tail and beard hairs; his expression stays firm and still, no smile; camera locked. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B08",
   "start": 31.7,
   "dur": 2.9,
   "kind": "place",
   "frame_prompt": "Night, a violent cold windstorm in a large desert army camp on an open plain: plain wool tents torn from their pegs, cooking pots overturned, campfires blown into streams of sparks, sand blowing hard from left to right. No people visible, no faces. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "Tents rip loose and flap away, sparks and sand stream hard across the frame from left to right, fires are blown out one by one; camera shaken slightly by the wind. Strong wind motion, no people, no text, no morphing, no flicker. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B09",
   "start": 34.6,
   "dur": 6.0,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Pale blue dawn. Medium close-up: he stands at the near edge of the trench looking across at the far plain, which is now empty: torn tent cloth, cold ash, hoof prints leading away. First light on his face. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "His eyes search the empty plain, brows lift in disbelief, then his shoulders drop as he lets out a long slow breath of relief; his eyes glisten; gentle push-in; no smile. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  },
  {
   "id": "B10",
   "start": 40.6,
   "dur": 6.4,
   "kind": "char",
   "frame_prompt": "Keep exactly the same fictional anonymous poor labourer, an Arab man of 30 from the Hijaz: deep tan uneven skin with fine scars and chapped patches, sunken cheeks, thick unkempt dark beard, dark tired eyes with heavy lower lids, a small pale scar through the left eyebrow; mended rough undyed cotton tunic with a darker patch on the right shoulder, frayed off-white wool head cloth wrapped simply with a loose tail on the left, bare forearms, a worn coil of palm-fibre rope over the left shoulder. Extreme close-up of his eyes and brow at sunrise, same framing as the night close-up; warm golden dawn light is reflected in both pupils instead of fire; the skin is warm-lit for the first time. Outskirts of Madinah, Hijaz, winter of 5 AH / 627 CE: black basalt ground, date palms, low mud-brick houses, squat stone towers; plain wool, cotton and leather only, no metal helmets, no gold, no domes, no minarets. Historically inspired cinematic hand-rendered pen-and-ink illustration with layered sepia and muted colour washes, visible directional pencil lines, fine crosshatching in the shadows, natural skin pores, creases, uneven pigmentation, flyaway beard hairs, woven-cloth fibres, plausible anatomy and human asymmetry. Not plastic, not glossy CGI, not anime, not cartoon. No text, no letters, no watermark, no border. Vertical 9:16.",
   "video_prompt": "Stillness: one slow blink, the eyes soften and shine with held-back tears that do not fall; a calm slow breath; imperceptible push-in. Subtle believable facial performance: slow natural breathing visible in the chest and nostrils, natural blinks, small eye movements, micro-tension in the jaw; cloth and beard hairs moving in the wind. Keep the face, scar, beard, head cloth and tunic exactly as in the first frame. No morphing, no flicker, no smile, no extra people, no text. Hand-drawn pen-and-ink illustration style kept throughout."
  }
 ]
}''')
OUT = Path('/kaggle/working/ahzab_out'); [ (OUT/d).mkdir(parents=True, exist_ok=True) for d in ['portraits','frames','shots','voice'] ]
DEV = 'cuda'
DT = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), DT)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('No Kaggle secret HF_TOKEN found:', e)
from huggingface_hub import login; login(os.environ['HF_TOKEN'])
def sheet(paths, out, cols=4, w=270):
    ims=[Image.open(p).convert('RGB') for p in paths]; h=int(w*ims[0].height/ims[0].width)
    rows=math.ceil(len(ims)/cols); s=Image.new('RGB',(cols*w,rows*h),'black')
    for i,im in enumerate(ims): s.paste(im.resize((w,h)),((i%cols)*w,(i//cols)*h))
    s.save(out); return s
def is_blank(img): return np.asarray(img).mean() < 4

## Stage A: portrait candidates

In [ ]:
from diffusers import FluxKontextPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TBnb
KONTEXT = 'black-forest-labs/FLUX.1-Kontext-dev'
def load_kontext():
    tr = FluxTransformer2DModel.from_pretrained(KONTEXT, subfolder='transformer', torch_dtype=DT,
            quantization_config=DBnb(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=DT))
    te = T5EncoderModel.from_pretrained(KONTEXT, subfolder='text_encoder_2', torch_dtype=DT,
            quantization_config=TBnb(load_in_8bit=True))
    p = FluxKontextPipeline.from_pretrained(KONTEXT, transformer=tr, text_encoder_2=te, torch_dtype=DT)
    p.enable_model_cpu_offload(); return p
kpipe = load_kontext()
paths=[]
for i in range(4):
    f = OUT/'portraits'/f'CHR-027_front_{i}.png'
    if not f.exists():
        img = kpipe(prompt=SHOTS['portrait_prompt'], height=1392, width=752, guidance_scale=3.5,
                    num_inference_steps=28, generator=torch.Generator('cpu').manual_seed(27000+i)).images[0]
        if is_blank(img): print('candidate', i, 'came out blank: switch DT to the other half-precision type and re-run')
        img.save(f)
    paths.append(f)
sheet(paths, OUT/'portraits'/'contact_sheet.png')

## STOP 1: approve the portrait
Look at the four candidates above (also in `ahzab_out/portraits/`). Put the number of the one you approve below, then run on.
The face, scar, beard, head cloth and tunic of this picture become the lock for every shot.

In [ ]:
CHOSEN = None   # e.g. 2
assert CHOSEN is not None, 'Pick a portrait number first'
PORTRAIT = Image.open(OUT/'portraits'/f'CHR-027_front_{CHOSEN}.png').convert('RGB')
PORTRAIT.save(OUT/'portraits'/'CHR-027_APPROVED.png')

## Stage B: first frame of every shot

In [ ]:
paths=[]
for s in SHOTS['shots']:
    f = OUT/'frames'/f"{s['id']}.png"
    if not f.exists():
        kw = dict(prompt=s['frame_prompt'], guidance_scale=2.5, num_inference_steps=28,
                  generator=torch.Generator('cpu').manual_seed(int(s['id'][1:])*101))
        if s['kind']=='char':
            img = kpipe(image=PORTRAIT, **kw).images[0]
        else:
            img = kpipe(height=1392, width=752, **kw).images[0]
        if is_blank(img): print(s['id'], 'came out blank')
        img.save(f)
    paths.append(f)
sheet(paths, OUT/'frames'/'contact_sheet.png', cols=5)

## STOP 2: check the frames
Every character frame must show **the same man** as the approved portrait (scar through the left eyebrow, patch on the right shoulder, head-cloth tail on the left).
To redo one frame: delete its file in `ahzab_out/frames/`, change its seed or prompt in `SHOTS`, and re-run the cell above. Then run on.

In [ ]:
del kpipe; gc.collect(); torch.cuda.empty_cache()

## Stage C: image-to-video shots

In [ ]:
from diffusers import LTXConditionPipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, BitsAndBytesConfig as TBnb
LTX = 'Lightricks/LTX-Video-0.9.7-distilled'
te = T5EncoderModel.from_pretrained(LTX, subfolder='text_encoder', torch_dtype=DT, quantization_config=TBnb(load_in_8bit=True))
vpipe = LTXConditionPipeline.from_pretrained(LTX, text_encoder=te, torch_dtype=DT)
vpipe.enable_model_cpu_offload(); vpipe.vae.enable_tiling()
W, H, FPS = 480, 832, 24
NEG = 'worst quality, blurry, jittery, distorted face, morphing, flicker, extra limbs, extra people, text, watermark, smiling, photorealistic, 3d render'
for s in SHOTS['shots']:
    f = OUT/'shots'/f"{s['id']}.mp4"
    if f.exists(): continue
    n = int(math.ceil(s['dur']*FPS/8))*8+1          # LTX wants 8k+1 frames
    img = Image.open(OUT/'frames'/f"{s['id']}.png").convert('RGB').resize((W,H), Image.LANCZOS)
    cond = LTXVideoCondition(image=img, frame_index=0)
    frames = vpipe(conditions=[cond], prompt=s['video_prompt'], negative_prompt=NEG, width=W, height=H,
                   num_frames=n, frame_rate=FPS, num_inference_steps=8, guidance_scale=1.0, decode_timestep=0.05,
                   decode_noise_scale=0.025, image_cond_noise_scale=0.0,
                   generator=torch.Generator('cpu').manual_seed(int(s['id'][1:])*7)).frames[0]
    export_to_video(frames, str(f), fps=FPS); print('made', f.name, n, 'frames')
    gc.collect(); torch.cuda.empty_cache()

## Stage D: Arabic narration

In [ ]:
import edge_tts
for n in SHOTS['narration']:
    f = OUT/'voice'/f"{n['id']}.mp3"
    if not f.exists():
        await edge_tts.Communicate(n['text'], SHOTS['narration_voice'], rate='-12%', pitch='-6Hz').save(str(f))
print(sorted(os.listdir(OUT/'voice')))

## Pack the results

In [ ]:
z = Path('/kaggle/working/ahzab_out.zip')
with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(OUT.parent))
print(z, round(z.stat().st_size/1e6,1), 'MB')